In [1]:
import matplotlib.pyplot as plt
import numpy as np
import tqdm

from ase.io import read
from mace.calculators import MACECalculator

from tensorpotential.calculator import TPCalculator


from sklearn.metrics import r2_score
import seaborn as sns
from sklearn.metrics import root_mean_squared_error

/home/johannes.karwounopoulos/miniconda3/envs/ai-fennel/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


2025-11-24 12:50:06.044247: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763985006.058381  774541 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763985006.063245  774541 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763985006.078496  774541 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763985006.078509  774541 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763985006.078511  774541 computation_placer.cc:177] computation placer alr

In [2]:
ref_structures = read("../data/b_off_2.0_all_energies.xyz", ":")
structures = read("../data/b_off_2.0_all_energies.xyz", ":")

# ref_structures = read("../data/b_off_2.0_ood_with_dft_total_energies.xyz", ":")
# structures = read("../data/b_off_2.0_ood_with_dft_total_energies.xyz", ":")

In [3]:
def calculate_energies_and_forces(model_type, layer, size, structures, ref_structures):

    pred_energies = []
    ref_energies = []
    pred_forces = []
    ref_forces = []

    if model_type.upper() == "MACE":
        if size == "medium":
            model_paths = f"../models/MACE-OFF24_{size}.model"
        else:
            model_paths = f"../models/MACE-OFF23_{size}.model"
        calc = MACECalculator(
            model_paths=model_paths,
            device="cuda",
        )
    elif model_type.upper() == "GRACE":
        calc = TPCalculator(model=f"../models/{layer}/b_off_{size}/seed/1/saved_model/")

    for i, struc in enumerate(tqdm.tqdm(structures)):

        if model_type.upper() == "MACE":
            struc.calc = calc
            pred_energies.append(struc.get_potential_energy())
            ref_energies.append(ref_structures[i].info["dft_total_energy"])

        elif model_type.upper() == "GRACE":
            struc.calc = calc
            pred_energies.append(struc.get_potential_energy())
            ref_energies.append(ref_structures[i].get_potential_energy())

        pred_forces.append(struc.get_forces())
        ref_forces.append(ref_structures[i].get_forces())

    return (
        pred_energies,
        ref_energies,
        pred_forces,
        ref_forces,
    )

In [4]:
energies = {}

model_types = ["mace"]
layers = ["1l"]
sizes = ["large","medium","small"]

for model_type in model_types:
    energies[model_type] = {}
    for layer in layers:
        energies[model_type][layer] = {}
        for size in sizes:
            energies[model_type][layer][size] = {}

            (
                energies[model_type][layer][size]["pred_energies"],
                energies[model_type][layer][size]["ref_energies"],
                energies[model_type][layer][size]["pred_forces"],
                energies[model_type][layer][size]["ref_forces"],
            ) = calculate_energies_and_forces(
                model_type,
                layer,
                size,
                structures,
                ref_structures,
            )

/home/johannes.karwounopoulos/miniconda3/envs/ai-fennel/lib/python3.11/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
No dtype selected, switching to float64 to match model dtype.


100%|██████████| 10000/10000 [04:05<00:00, 40.65it/s]
/home/johannes.karwounopoulos/miniconda3/envs/ai-fennel/lib/python3.11/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
No dtype selected, switching to float64 to match model dtype.


100%|██████████| 10000/10000 [03:26<00:00, 48.39it/s]
/home/johannes.karwounopoulos/miniconda3/envs/ai-fennel/lib/python3.11/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
No dtype selected, switching to float64 to match model dtype.


100%|██████████| 10000/10000 [02:50<00:00, 58.71it/s]


In [5]:
for prop in ["energies", "forces"]:
    print(f"##### {prop} results #####")
    for model_type in model_types:
        for layer in layers:
            for size in sizes:

                if prop == "forces":
                    predicted = np.concatenate(
                        energies[model_type][layer][size][f"pred_{prop}"]
                    )
                    reference = np.concatenate(
                        energies[model_type][layer][size][f"ref_{prop}"]
                    )
                else:
                    predicted = np.array(
                        energies[model_type][layer][size][f"pred_{prop}"]
                    )
                    reference = np.array(
                        energies[model_type][layer][size][f"ref_{prop}"]
                    )

                mae = np.mean(np.abs(predicted - reference))
                rmse = root_mean_squared_error(reference, predicted)
                r2 = r2_score(reference, predicted)

                # Store metrics in the dictionary
                energies[model_type][layer][size][f"total_{prop}_mae"] = mae
                energies[model_type][layer][size][f"total_{prop}_rmse"] = rmse
                energies[model_type][layer][size][f"total_{prop}_r2"] = r2

                print(
                    f"Model: {model_type.upper()}, Layer: {layer}, Size: {size}, MAE: {mae*1000:.4f} meV, RMSE: {rmse*1000:.4f} meV, R²: {r2:.4f}"
                )

# Manually add pre-computed MACE total metrics
# The values are divided by 1000 to convert from meV to eV
# if "mace" not in energies:
#     energies["mace"] = {"N/A": {"small": {}, "medium": {}}}

# energies["mace"]["N/A"]["small"]["total_energies_mae"] = 43.0719 / 1000
# energies["mace"]["N/A"]["small"]["total_energies_rmse"] = 88.3665 / 1000
# energies["mace"]["N/A"]["small"]["total_energies_r2"] = 1.0000
# energies["mace"]["N/A"]["small"]["total_forces_mae"] = 31.9850 / 1000
# energies["mace"]["N/A"]["small"]["total_forces_rmse"] = 64.0469 / 1000
# energies["mace"]["N/A"]["small"]["total_forces_r2"] = 0.9961

# energies["mace"]["N/A"]["medium"]["total_energies_mae"] = 26.9742 / 1000
# energies["mace"]["N/A"]["medium"]["total_energies_rmse"] = 54.2475 / 1000
# energies["mace"]["N/A"]["medium"]["total_energies_r2"] = 1.0000
# energies["mace"]["N/A"]["medium"]["total_forces_mae"] = 20.6761 / 1000
# energies["mace"]["N/A"]["medium"]["total_forces_rmse"] = 47.7470 / 1000
# energies["mace"]["N/A"]["medium"]["total_forces_r2"] = 0.9978

print("\nMACE total metrics loaded into the `energies` dictionary.")


def print_mace_results():
    print(
        """#####################################
#### MACE results for comparison ####
##### energies results ##############
Model: MACE, Size: small, MAE: 43.0719 meV, RMSE: 88.3665 meV, R²: 1.0000
Model: MACE, Size: medium, MAE: 26.9742 meV, RMSE: 54.2475 meV, R²: 1.0000
##### forces results #####
Model: MACE, Size: small, MAE: 31.9850 meV, RMSE: 64.0469 meV, R²: 0.9961
Model: MACE, Size: medium, MAE: 20.6761 meV, RMSE: 47.7470 meV, R²: 0.9978"""
    )


# print_mace_results()

##### energies results #####
Model: MACE, Layer: 1l, Size: large, MAE: 25.0061 meV, RMSE: 57.5306 meV, R²: 1.0000
Model: MACE, Layer: 1l, Size: medium, MAE: 26.9742 meV, RMSE: 54.2475 meV, R²: 1.0000
Model: MACE, Layer: 1l, Size: small, MAE: 43.0719 meV, RMSE: 88.3665 meV, R²: 1.0000
##### forces results #####
Model: MACE, Layer: 1l, Size: large, MAE: 13.6198 meV, RMSE: 41.3304 meV, R²: 0.9984
Model: MACE, Layer: 1l, Size: medium, MAE: 20.6761 meV, RMSE: 47.7360 meV, R²: 0.9978
Model: MACE, Layer: 1l, Size: small, MAE: 31.9850 meV, RMSE: 64.0362 meV, R²: 0.9961

MACE total metrics loaded into the `energies` dictionary.


In [6]:
energies["nr_of_atoms"] = []
for struc in structures:
    energies["nr_of_atoms"].append(len(struc))

In [7]:
for model_type in model_types:
    print(f"##### {model_type.upper()} results #####")
    for layer in layers:
        for size in sizes:

            print(f"--- {layer} layer, {size} size ---")
            pred_E = np.asarray(energies[model_type][layer][size]["pred_energies"])
            ref_E = np.asarray(energies[model_type][layer][size]["ref_energies"])
            natoms = np.asarray(energies["nr_of_atoms"][0 : len(pred_E)])

            # Calculate energy metrics
            energy_rmse_per_atom = np.sqrt(np.mean(((pred_E - ref_E) / natoms) ** 2))
            energy_mae_per_atom = np.mean(np.abs((pred_E - ref_E) / natoms))

            # Store energy metrics
            energies[model_type][layer][size][
                "energy_rmse_per_atom"
            ] = energy_rmse_per_atom
            energies[model_type][layer][size][
                "energy_mae_per_atom"
            ] = energy_mae_per_atom

            print("Energy RMSE per atom (meV):", f"{energy_rmse_per_atom * 1000:.2f}")
            print("Energy MAE per atom (meV):", f"{energy_mae_per_atom * 1000:.2f}")

            pred_list = energies[model_type][layer][size]["pred_forces"]
            ref_list = energies[model_type][layer][size]["ref_forces"]

            # stack diffs over all molecules
            diffs = [np.asarray(p) - np.asarray(r) for p, r in zip(pred_list, ref_list)]
            D = np.concatenate(diffs, axis=0)  # shape (sum_i N_i, 3)

            force_rmse_per_atom = np.sqrt(np.mean(D**2))  # eV/Å
            force_mae_per_atom = np.mean(np.abs(D))  # eV/Å

            # Store force metrics
            energies[model_type][layer][size][
                "force_rmse_per_atom"
            ] = force_rmse_per_atom
            energies[model_type][layer][size]["force_mae_per_atom"] = force_mae_per_atom

            print(f"Forces RMSE per atom (meV/Å): {force_rmse_per_atom*1000:.4f}")
            print(f"Forces MAE per atom (meV/Å):  {force_mae_per_atom*1000:.4f}")

# Manually add pre-computed MACE results
# The values are divided by 1000 to convert from meV to eV, to match the units of the GRACE calculations
# energies["mace"] = {
#     "N/A": {
#         "small": {
#             "energy_rmse_per_atom": 2.54 / 1000,
#             "energy_mae_per_atom": 1.26 / 1000,
#             "force_rmse_per_atom": 64.0469 / 1000,
#             "force_mae_per_atom": 31.9850 / 1000,
#         },
#         "medium": {
#             "energy_rmse_per_atom": 1.87 / 1000,
#             "energy_mae_per_atom": 0.81 / 1000,
#             "force_rmse_per_atom": 47.7470 / 1000,
#             "force_mae_per_atom": 20.6761 / 1000,
#         },
#     }
# }
# print(
#     """#######################################
# ##### MACE results for comparison #####
# --- SMALL model ---
# Energy RMSE per atom (meV): 2.54
# Energy MAE per atom (meV): 1.26
# Forces RMSE per atom (meV/Å): 64.0469
# Forces MAE per atom (meV/Å):  31.9850
# --- MEDIUM model ---
# Energy RMSE per atom (meV): 1.87
# Energy MAE per atom (meV): 0.81
# Forces RMSE per atom (meV/Å): 47.7470
# Forces MAE per atom (meV/Å):  20.6761"""
# )

##### MACE results #####
--- 1l layer, large size ---
Energy RMSE per atom (meV): 2.73
Energy MAE per atom (meV): 0.79
Forces RMSE per atom (meV/Å): 41.4044
Forces MAE per atom (meV/Å):  13.6198
--- 1l layer, medium size ---
Energy RMSE per atom (meV): 1.87
Energy MAE per atom (meV): 0.81
Forces RMSE per atom (meV/Å): 47.7470
Forces MAE per atom (meV/Å):  20.6761
--- 1l layer, small size ---
Energy RMSE per atom (meV): 2.54
Energy MAE per atom (meV): 1.26
Forces RMSE per atom (meV/Å): 64.0469
Forces MAE per atom (meV/Å):  31.9850


In [ ]:
import seaborn as sns

# Prepare data for plotting
model_labels = []
energy_rmse_values = []
energy_mae_values = []
force_rmse_values = []
force_mae_values = []

for model_type in model_types:
    for layer in layers:
        for size in sizes:
            label = f"{layer.upper()}-{model_type}\n{size}"
            model_labels.append(label)

            energy_rmse_values.append(
                energies[model_type][layer][size]["energy_rmse_per_atom"] * 1000
            )
            energy_mae_values.append(
                energies[model_type][layer][size]["energy_mae_per_atom"] * 1000
            )
            force_rmse_values.append(
                energies[model_type][layer][size]["force_rmse_per_atom"] * 1000
            )
            force_mae_values.append(
                energies[model_type][layer][size]["force_mae_per_atom"] * 1000
            )

# Define colors based on model size (not individual models)
model_colors_list = []
size_colors = {
    'small': '#1f77b4',    # Blue
    'medium': '#ff7f0e',   # Orange
    'large': '#2ca02c'     # Green
}

# Assign colors based on size for each model
for model_type in model_types:
    for layer in layers:
        for size in sizes:
            model_colors_list.append(size_colors[size])

# Create the plots with Seaborn
fig, axs = plt.subplots(2, 2, figsize=(8, 5))

# --- MACE reference values ---
mace_metrics = {
    "small": {
        "energy_rmse": energies["mace"]["N/A"]["small"]["energy_rmse_per_atom"] * 1000,
        "energy_mae": energies["mace"]["N/A"]["small"]["energy_mae_per_atom"] * 1000,
        "force_rmse": energies["mace"]["N/A"]["small"]["force_rmse_per_atom"] * 1000,
        "force_mae": energies["mace"]["N/A"]["small"]["force_mae_per_atom"] * 1000,
    },
    "medium": {
        "energy_rmse": energies["mace"]["N/A"]["medium"]["energy_rmse_per_atom"] * 1000,
        "energy_mae": energies["mace"]["N/A"]["medium"]["energy_mae_per_atom"] * 1000,
        "force_rmse": energies["mace"]["N/A"]["medium"]["force_rmse_per_atom"] * 1000,
        "force_mae": energies["mace"]["N/A"]["medium"]["force_mae_per_atom"] * 1000,
    },
}


# Energy RMSE per atom
sns.barplot(
    ax=axs[0, 0],
    x=model_labels,
    y=energy_rmse_values,
    hue=model_labels,
    palette=model_colors_list,
    legend=False,
)
axs[0, 0].axhline(
    y=mace_metrics["small"]["energy_rmse"],
    color="gray",
    linestyle="--",
    label="MACE Small",
)
axs[0, 0].axhline(
    y=mace_metrics["medium"]["energy_rmse"],
    color="black",
    linestyle="--",
    label="MACE Medium",
)
axs[0, 0].set_title("Energy RMSE per Atom")
axs[0, 0].set_ylabel("RMSE (meV/atom)")
axs[0, 0].set_xlabel("")

# Energy MAE per atom
sns.barplot(
    ax=axs[0, 1],
    x=model_labels,
    y=energy_mae_values,
    hue=model_labels,
    palette=model_colors_list,
    legend=False,
)
axs[0, 1].axhline(
    y=mace_metrics["small"]["energy_mae"],
    color="gray",
    linestyle="--",
    label="MACE Small",
)
axs[0, 1].axhline(
    y=mace_metrics["medium"]["energy_mae"],
    color="black",
    linestyle="--",
    label="MACE Medium",
)
axs[0, 1].set_title("Energy MAE per Atom")
axs[0, 1].set_ylabel("MAE (meV/atom)")
axs[0, 1].set_xlabel("")

# Force RMSE per atom
sns.barplot(
    ax=axs[1, 0],
    x=model_labels,
    y=force_rmse_values,
    hue=model_labels,
    palette=model_colors_list,
    legend=False,
)
axs[1, 0].axhline(
    y=mace_metrics["small"]["force_rmse"],
    color="gray",
    linestyle="--",
    label="MACE Small",
)
axs[1, 0].axhline(
    y=mace_metrics["medium"]["force_rmse"],
    color="black",
    linestyle="--",
    label="MACE Medium",
)
axs[1, 0].set_title("Force RMSE per Atom")
axs[1, 0].set_ylabel("RMSE (meV/Å)")
axs[1, 0].set_xlabel("")

# Force MAE per atom
sns.barplot(
    ax=axs[1, 1],
    x=model_labels,
    y=force_mae_values,
    hue=model_labels,
    palette=model_colors_list,
    legend=False,
)
axs[1, 1].axhline(
    y=mace_metrics["small"]["force_mae"],
    color="gray",
    linestyle="--",
    label="MACE Small",
)
axs[1, 1].axhline(
    y=mace_metrics["medium"]["force_mae"],
    color="black",
    linestyle="--",
    label="MACE Medium",
)
axs[1, 1].set_title("Force MAE per Atom")
axs[1, 1].set_ylabel("MAE (meV/Å)")
axs[1, 1].set_xlabel("")
axs[0, 0].legend()

fig.suptitle("B_off model", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("a_wpS_bar.png", dpi=300)
plt.show()


In [ ]:
# import pickle

# with open('energies.pkl', 'wb') as f:
#     pickle.dump(energies, f)